# Fast4DReg Python

Minimal notebook interface for projection-based time or channel registration. Registration logic lives in the `fast4dreg` package.

In [ ]:
# Uncomment in Google Colab after the package is published or replace with the GitHub URL.
# %pip install -q 'fast4dreg-python[notebook]'

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import tifffile

from fast4dreg import (
    evaluate_registration, register_channels, register_time,
    save_drift_csv, save_tiff, save_validation_csv,
)

print('Fast4DReg Python ready')

## Registration app

Enter a TIFF path accessible to the notebook. For Colab, mount Google Drive first and use a Drive path.

In [ ]:
image_path = widgets.Text(description='TIFF path:', layout=widgets.Layout(width='700px'))
axes = widgets.Text(value='TZYX', description='Axes:')
mode = widgets.Dropdown(options=['time', 'channels'], value='time', description='Mode:')
reference_channel = widgets.IntText(value=0, description='Ref channel:')
projection = widgets.Dropdown(options=['max', 'mean'], value='max', description='Projection:')
reference = widgets.Dropdown(options=['first', 'previous'], value='first', description='Reference:')
correct_xy = widgets.Checkbox(value=True, description='Correct XY')
correct_z = widgets.Checkbox(value=False, description='Correct Z')
backend = widgets.Dropdown(options=['threaded', 'unthreaded'], value='threaded', description='Backend:')
workers = widgets.BoundedIntText(value=4, min=1, max=64, description='Workers:')
overwrite = widgets.Checkbox(value=False, description='Overwrite outputs')
validate = widgets.Checkbox(value=True, description='Validate time registration')
validation_analysis = widgets.Dropdown(options=['slice', 'max', 'mean'], value='slice', description='Validation:')
run_button = widgets.Button(description='Run registration', button_style='primary')
output = widgets.Output()

def run_registration(_):
    with output:
        output.clear_output()
        path = Path(image_path.value).expanduser()
        if not path.exists():
            raise FileNotFoundError(f'Image not found: {path}')

        image = tifffile.imread(path)
        settings = dict(
            axes=axes.value.upper(),
            reference_channel=reference_channel.value,
            projection=projection.value,
            correct_xy=correct_xy.value,
            correct_z=correct_z.value,
            backend=backend.value,
            workers=workers.value,
        )
        if mode.value == 'time':
            corrected, shifts = register_time(
                image, reference=reference.value, **settings
            )
        else:
            corrected, shifts = register_channels(image, **settings)

        output_tiff = path.with_name(path.stem + '_registered.tif')
        output_csv = path.with_name(path.stem + '_drift.csv')
        planned_outputs = [output_tiff, output_csv]
        if mode.value == 'time' and validate.value:
            planned_outputs.extend([
                path.with_name(path.stem + '_validation_before.csv'),
                path.with_name(path.stem + '_validation_after.csv'),
            ])
        existing = [output_path for output_path in planned_outputs if output_path.exists()]
        if existing and not overwrite.value:
            raise FileExistsError(f'Outputs already exist: {existing}. Enable Overwrite outputs to replace them.')
        save_tiff(output_tiff, corrected, axes=axes.value, overwrite=overwrite.value)
        save_drift_csv(output_csv, shifts, overwrite=overwrite.value)
        if mode.value == 'time' and validate.value:
            validation_options = dict(
                axes=axes.value.upper(),
                reference=reference.value,
                reference_channel=reference_channel.value,
                analysis=validation_analysis.value,
            )
            before = evaluate_registration(image, **validation_options)
            after = evaluate_registration(corrected, **validation_options)
            before_csv = path.with_name(path.stem + '_validation_before.csv')
            after_csv = path.with_name(path.stem + '_validation_after.csv')
            save_validation_csv(before_csv, before, overwrite=overwrite.value)
            save_validation_csv(after_csv, after, overwrite=overwrite.value)

        labels = ('dz', 'dy', 'dx')
        for index, label in enumerate(labels):
            plt.plot(shifts[:, index], label=label)
        plt.xlabel('Timepoint' if mode.value == 'time' else 'Channel')
        plt.ylabel('Translation to apply (pixels)')
        plt.legend()
        plt.show()
        print(f'Saved: {output_tiff}')
        print(f'Saved: {output_csv}')
        if mode.value == 'time' and validate.value:
            print(f'Mean mSSIM: {sum(row.mssim for row in before) / len(before):.4f} -> {sum(row.mssim for row in after) / len(after):.4f}')
            print(f'Mean Pearson: {sum(row.pearson for row in before) / len(before):.4f} -> {sum(row.pearson for row in after) / len(after):.4f}')
            print(f'Saved: {before_csv}')
            print(f'Saved: {after_csv}')

run_button.on_click(run_registration)
display(widgets.VBox([
    image_path, axes, mode, reference_channel, projection, reference, backend, workers,
    widgets.HBox([correct_xy, correct_z]),
    widgets.HBox([validate, validation_analysis, overwrite]), run_button, output
]))